In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import torch
import yaml
from tqdm import tqdm
from chronos import BaseChronosPipeline, Chronos2Pipeline

warnings.filterwarnings("ignore")

In [3]:
CONFIG_PATH = "config.yaml"
DATASET = "cpcb"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)[DATASET]["chronos"]

pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained(
    cfg["model_path"], device_map="cuda"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

In [4]:
POL_MAP = {
    "SO2":   "SO2 (µg/m³)",
    "PM2.5": "PM2.5 (µg/m³)",
    "Ozone": "Ozone (µg/m³)",
    "NO2":   "NO2 (µg/m³)",
    "CO":    "CO (mg/m³)",
    "PM10":  "PM10 (µg/m³)",
}


def make_windows(df, timestamp_col, target_col, context_length, prediction_length):
    """Sliding window over a single-site DataFrame. Returns list of dicts."""
    values     = df[target_col].values
    timestamps = df[timestamp_col].values
    total      = context_length + prediction_length
    windows = []
    for i in range(len(df) - total + 1):
        windows.append({
            "past_values":   values[i : i + context_length],
            "future_values": values[i + context_length : i + total],
            "timestamp":     timestamps[i + context_length - 1],
        })
    return windows


def build_window_df(windows, target_col, timestamp_col, context_length):
    """Convert windows list into a long-form DataFrame for Chronos predict_df."""
    rows = []
    for window_id, w in enumerate(windows):
        timestamps = pd.date_range(end=w["timestamp"], periods=context_length, freq="h")
        for t, val in zip(timestamps, w["past_values"]):
            rows.append({"item_id": window_id, timestamp_col: t, target_col: val})
    return pd.DataFrame(rows)


def zeroshot_eval(df, cfg, target_col, pipeline):
    timestamp_col     = cfg["timestamp_column"]
    context_length    = cfg["context_length"]
    prediction_length = cfg["prediction_length"]
    batch_size        = cfg["batch_size"]

    windows   = make_windows(df, timestamp_col, target_col, context_length, prediction_length)
    window_df = build_window_df(windows, target_col, timestamp_col, context_length)

    pred_df = pipeline.predict_df(
        window_df,
        prediction_length=prediction_length,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="item_id",
        timestamp_column=timestamp_col,
        target=target_col,
        batch_size=batch_size,
    )
    return pred_df

In [5]:
# --- Single file test ---
input_dir         = cfg["input_dir"]
timestamp_col     = cfg["timestamp_column"]
context_length    = cfg["context_length"]
prediction_length = cfg["prediction_length"]

test_file = sorted(f for f in os.listdir(input_dir) if os.path.isfile(os.path.join(input_dir, f)))[0]
site_name = os.path.splitext(test_file)[0]
target_col = POL_MAP[site_name.split("_")[-1]]

print(f"File:   {test_file}")
print(f"Target: {target_col}")

# 1. Load CSV
df = pd.read_csv(os.path.join(input_dir, test_file), parse_dates=[timestamp_col])
print(f"\nRaw df: {df.shape}")
display(df.head(3))

# 2. Make windows
windows = make_windows(df, timestamp_col, target_col, context_length, prediction_length)
print(f"\nWindows: {len(windows)}  (context={context_length}, horizon={prediction_length})")
print(f"Window[0] keys:      {list(windows[0].keys())}")
print(f"past_values shape:   {windows[0]['past_values'].shape}")
print(f"future_values shape: {windows[0]['future_values'].shape}")

# 3. Build long-form df
window_df = build_window_df(windows, target_col, timestamp_col, context_length)
print(f"\nwindow_df: {window_df.shape}")
display(window_df.head(5))

# 4. Chronos predict
print("\nRunning Chronos predict_df ...")
pred_df = zeroshot_eval(df, cfg, target_col, pipeline)
print(f"pred_df: {pred_df.shape}")
display(pred_df.head(10))

File:   site_105_North_Campus_DU_Delhi_IMD_CO.csv
Target: CO (mg/m³)

Raw df: (26304, 2)


,Timestamp,CO (mg/m³)
0,2022-07-01 00:00:00,3.93
1,2022-07-01 01:00:00,3.43
2,2022-07-01 02:00:00,2.02



Windows: 26125  (context=168, horizon=12)
Window[0] keys:      ['past_values', 'future_values', 'timestamp']
past_values shape:   (168,)
future_values shape: (12,)

window_df: (4389000, 3)


,item_id,Timestamp,CO (mg/m³)
0,0,2022-07-01 00:00:00,3.93
1,0,2022-07-01 01:00:00,3.43
2,0,2022-07-01 02:00:00,2.02
3,0,2022-07-01 03:00:00,0.27
4,0,2022-07-01 04:00:00,0.04



Running Chronos predict_df ...
pred_df: (313500, 7)


,item_id,Timestamp,target_name,predictions,0.1,0.5,0.9
0,0,2022-07-08 00:00:00,CO (mg/m³),2.278661,1.889699,2.278661,2.798331
1,0,2022-07-08 01:00:00,CO (mg/m³),2.154800,1.733240,2.154800,2.741647
2,0,2022-07-08 02:00:00,CO (mg/m³),2.046489,1.596066,2.046489,2.636624
3,0,2022-07-08 03:00:00,CO (mg/m³),1.984468,1.509813,1.984468,2.612986
4,0,2022-07-08 04:00:00,CO (mg/m³),1.963651,1.479934,1.963651,2.603974
5,0,2022-07-08 05:00:00,CO (mg/m³),1.995811,1.481775,1.995811,2.671489
6,0,2022-07-08 06:00:00,CO (mg/m³),2.036878,1.484145,2.036878,2.755152
7,0,2022-07-08 07:00:00,CO (mg/m³),2.053719,1.512369,2.053719,2.787794
8,0,2022-07-08 08:00:00,CO (mg/m³),2.050778,1.481190,2.050778,2.824756
9,0,2022-07-08 09:00:00,CO (mg/m³),2.037955,1.451147,2.037955,2.821385


In [ ]:
input_dir  = cfg["input_dir"]
output_dir = cfg["output_dir"]
context_length    = cfg["context_length"]
prediction_length = cfg["prediction_length"]
timestamp_col     = cfg["timestamp_column"]
os.makedirs(output_dir, exist_ok=True)

files = sorted(f for f in os.listdir(input_dir) if os.path.isfile(os.path.join(input_dir, f)))

for file in tqdm(files, desc="Processing sites"):
    site_name = os.path.splitext(file)[0]
    site_dir  = os.path.join(output_dir, site_name)
    os.makedirs(site_dir, exist_ok=True)

    target_col = POL_MAP[site_name.split("_")[-1]]
    df = pd.read_csv(os.path.join(input_dir, file), parse_dates=[timestamp_col])

    # ── Windows ──────────────────────────────────────────────────────────────
    windows   = make_windows(df, timestamp_col, target_col, context_length, prediction_length)
    window_df = build_window_df(windows, target_col, timestamp_col, context_length)

    # ── Chronos predict ───────────────────────────────────────────────────────
    pred_df = pipeline.predict_df(
        window_df,
        prediction_length=prediction_length,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="item_id",
        timestamp_column=timestamp_col,
        target=target_col,
        batch_size=cfg["batch_size"],
    )

    # ── Build tensors (N, T, 1) to match TTM output format ───────────────────
    N = len(windows)

    past_np   = np.stack([w["past_values"]   for w in windows]).astype(np.float32)  # (N, 168)
    future_np = np.stack([w["future_values"] for w in windows]).astype(np.float32)  # (N, 12)

    dataset_tensors = {
        "past_values":   torch.tensor(past_np).unsqueeze(-1),   # (N, 168, 1)
        "future_values": torch.tensor(future_np).unsqueeze(-1), # (N, 12, 1)
    }

    # Pivot pred_df: sort by item_id + Timestamp, then reshape to (N, 12)
    preds_np = (
        pred_df.sort_values(["item_id", timestamp_col])
               .groupby("item_id")["predictions"]
               .apply(np.array)
               .values
    )
    preds_np     = np.stack(preds_np).astype(np.float32)              # (N, 12)
    preds_tensor = torch.tensor(preds_np).unsqueeze(-1)               # (N, 12, 1)

    timestamps = np.array([w["timestamp"] for w in windows])          # (N,)

    # ── Save ──────────────────────────────────────────────────────────────────
    torch.save(dataset_tensors, os.path.join(site_dir, "dataset.pt"))
    torch.save(preds_tensor,    os.path.join(site_dir, "predictions.pt"))
    torch.save(timestamps,      os.path.join(site_dir, "timestamps.pt"))

print(f"\nAll results saved to {output_dir}")